In [424]:
# Importy
import os, json, pickle, math, random, re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pretty_midi
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR
#from tqdm.notebook import tqdm 

from functions import DoomDataset

In [425]:
# Parametry
VOCAB_SIZE = 579
EMBEDDING_DIM = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 2
STRIDE = 64
CONTEXT = 192
BATCH_SIZE = 256
EPOCHS = 42
LR = 0.002
MAX_TRANSPOSE = 12
STEPS_PER_BEAT = 12
# Zmienic przed odpaleniem treningu. 
# Wszyscy wiemy, że zajmująca maks 5 minut implementacja prostego sprawdzania, czy plik istnieje jest zbędna i niepotrzebnie komplikuje kod prawda?
FILE_NAME_SAVE = 'lstm_music_model_42ep_2Layers_512hidden_lr0001_context192_ed256_stride64_bs256__05d1_03d2_V28.pth'
FILE_NAME_LOAD = 'lstm_music_model_42ep_2Layers_512hidden_lr0001_context192_ed256_stride64_bs256__05d1_03d2_V28.pth'

#lstm_music_model_50ep_2Layers_512hidden_lr0005_context512_ed512_stride256_V7.pth
#  lstm_music_model_60ep_2Layers_512hidden_lr0005_context512_ed512_stride256_V9
# best V13/14/16_J
os.makedirs('../models', exist_ok=True)
os.makedirs('../output', exist_ok=True)
os.makedirs('../output/LSTM', exist_ok=True)

PROC_DIR = '../data/processed/'
OUTPUT_DIR = '../output/LSTM/'
MODEL_DIR = '../models/'

In [426]:
# Ustwaianie ziarna i cuda
random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Device: NVIDIA GeForce RTX 4060 Laptop GPU


In [427]:
# Ładowanie danych i słownika tokenów
with open(os.path.join(PROC_DIR, 'sequences.pkl'), 'rb') as f:
    data = pickle.load(f)

with open(os.path.join(PROC_DIR, 'vocab.json')) as f:
    token2id = json.load(f)
id2token = {y: x for x,y in token2id.items()}

sequences = data.values()
names = list(data.keys())
VOCAB_SIZE = len(token2id)
PAD_ID = token2id['PAD']
print(f'Sequences: {len(sequences)} | Vocab: {VOCAB_SIZE} | Tokens: {sum(len(s) for s in sequences):,}')

Sequences: 122 | Vocab: 579 | Tokens: 1,253,064


In [428]:
def music_transpose_collate_fn(batch, token2id, id2token, max_transpose=12):
    shift = random.randint(-max_transpose, max_transpose)
    if shift == 0:
        shift = 1
        
    transposed_features_batch = []
    labels_batch = []
    
    for sample in batch:
        sequence, label = sample
        
        if isinstance(sequence, torch.Tensor):
            sequence_list = sequence.tolist()
        else:
            sequence_list = list(sequence)
            
        transposed_sequence = []
        
        for token_id in sequence_list:
            token_str = id2token[token_id]
            
            if any(p in token_str.lower() for p in ["perc", "drum", "bos", "eos", "pad", "time"]):
                transposed_sequence.append(token_id)
                continue
                
            match = re.search(r'\d+', token_str)
            if match:
                try:
                    pitch = int(match.group())
                    
                    if 0 <= pitch <= 127:
                        new_pitch = max(0, min(127, pitch + shift))
                        new_token_str = token_str.replace(str(pitch), str(new_pitch))
                        
                        if new_token_str in token2id:
                            transposed_sequence.append(token2id[new_token_str])
                            continue
                except ValueError:
                    pass
            
            transposed_sequence.append(token_id)
            
        transposed_features_batch.append(torch.tensor(transposed_sequence, dtype=torch.long))
        
        if isinstance(label, torch.Tensor):
            labels_batch.append(label.clone().detach())
        else:
            labels_batch.append(torch.tensor(label, dtype=torch.long))
        
    return torch.stack(transposed_features_batch, dim=0), torch.stack(labels_batch, dim=0)


In [429]:
# Przygotowanie DataLoaderów
sequences_list = list(sequences)
random.shuffle(sequences_list)
train_sequences, val_sequences = sequences_list[:int(0.9*len(sequences_list))], sequences_list[int(0.9*len(sequences_list)):]
train_dataset = DoomDataset(train_sequences, context = CONTEXT, stride = STRIDE, augment=True, max_transpose=MAX_TRANSPOSE)
print(f"Total training samples generated: {len(train_dataset)}")
print(f"With batch_size={BATCH_SIZE}, one epoch will take {len(train_dataset) // BATCH_SIZE} steps.")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,)

val_dataset = DoomDataset(val_sequences, context = CONTEXT, stride=STRIDE, augment=False, max_transpose=MAX_TRANSPOSE)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Total validation samples generated: {len(val_dataset)}")

Total training samples generated: 17110
With batch_size=256, one epoch will take 66 steps.
Total validation samples generated: 2161


In [430]:
# Pobranie jednej paczki danych
features, labels = next(iter(train_loader))

# Sprawdzenie kształtu danych
print(f"Kształt cech (X): {features.shape}")
print(f"Kształt etykiet (y): {labels.shape}")


Kształt cech (X): torch.Size([256, 192])
Kształt etykiet (y): torch.Size([256, 192])


In [431]:
# Definicja modelu
class MusicLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super(MusicLSTM, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers, batch_first=True, dropout=0.5)
        
        self.ln = nn.LayerNorm(hidden_size)

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x, hidden=None):
        
        embedded = self.embedding(x)
        #dodane od v24
        embedded = self.dropout(embedded)

        out, hidden = self.lstm(embedded, hidden)
        #usunięte w v24
        #out = self.ln(out)

        out = self.dropout(out)

        logits = self.fc(out)
        
        return logits, hidden
  

In [432]:
# Inicjalizacja modelu, optymalizatora i funkcji straty
model = MusicLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS).to(device)

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.15) # Poza testami zostawić na 0.1
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.02)


In [433]:
# Trenowanie modelu
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=8, factor=0.5)

for epoch in range(EPOCHS):
   
    model.train()
    total_loss = 0
    total_batches = len(train_loader)

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()

        logits, _ = model(inputs)

        loss = loss_fn(logits.view(-1, VOCAB_SIZE), targets.view(-1))
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 40 == 0 or batch_idx == total_batches - 1:
            current_avg_loss = total_loss / (batch_idx + 1)
            print(f"Epoch: [{epoch+1}/{EPOCHS}] | Batch: [{batch_idx + 1}/{total_batches}] | Train Loss: {current_avg_loss:.4f}")

    epoch_train_loss = total_loss / total_batches
    print(f"Epoch {epoch+1} Train Summary: Avg Loss = {epoch_train_loss:.4f} ")

    model.eval() 
    total_val_loss = 0
    val_batches = len(val_loader)

    with torch.no_grad():
        for val_inputs, val_targets in val_loader:
            val_inputs = val_inputs.to(device)
            val_targets = val_targets.to(device)

            val_logits, _ = model(val_inputs)
            val_loss = loss_fn(val_logits.view(-1, VOCAB_SIZE), val_targets.view(-1))
            
            total_val_loss += val_loss.item()

    epoch_val_loss = total_val_loss / val_batches
    print(f"Epoch {epoch+1} Val Summary: Avg Loss = {epoch_val_loss:.4f}")

    scheduler.step(epoch_val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Learning rate for next epoch: {current_lr}\n")


Epoch: [1/42] | Batch: [1/67] | Train Loss: 6.3611
Epoch: [1/42] | Batch: [41/67] | Train Loss: 5.2568
Epoch: [1/42] | Batch: [67/67] | Train Loss: 4.9062
Epoch 1 Train Summary: Avg Loss = 4.9062 
Epoch 1 Val Summary: Avg Loss = 4.2684
Learning rate for next epoch: 0.002

Epoch: [2/42] | Batch: [1/67] | Train Loss: 4.1698
Epoch: [2/42] | Batch: [41/67] | Train Loss: 3.9126
Epoch: [2/42] | Batch: [67/67] | Train Loss: 3.7933
Epoch 2 Train Summary: Avg Loss = 3.7933 
Epoch 2 Val Summary: Avg Loss = 3.7781
Learning rate for next epoch: 0.002

Epoch: [3/42] | Batch: [1/67] | Train Loss: 3.4819
Epoch: [3/42] | Batch: [41/67] | Train Loss: 3.4031
Epoch: [3/42] | Batch: [67/67] | Train Loss: 3.3494
Epoch 3 Train Summary: Avg Loss = 3.3494 
Epoch 3 Val Summary: Avg Loss = 3.5819
Learning rate for next epoch: 0.002

Epoch: [4/42] | Batch: [1/67] | Train Loss: 3.2022
Epoch: [4/42] | Batch: [41/67] | Train Loss: 3.1481
Epoch: [4/42] | Batch: [67/67] | Train Loss: 3.1203
Epoch 4 Train Summary: Avg

KeyboardInterrupt: 

In [434]:
# Zapis modelu
torch.save(model.state_dict(), os.path.join(MODEL_DIR, FILE_NAME_SAVE))

In [435]:
def is_not_percusion(token):
    return True if token < 259 or token > 385 else False
def is_note(token):
    return True if token > 2 and token < 131 else False

In [467]:
# Nowy sposób na generację
def generate_music_cerative(model, token2id, id2token, max_generated_tokens=2000, temperature=0.65):
    model.eval()

    generated = [token2id['BOS']]
    hidden = None
    last_active_program = None
    
    in_creative_mode = False
    tokens_left_in_mode = random.randint(600, 1000) 

    with torch.no_grad():
        for i in range(max_generated_tokens):
            input_tensor = torch.tensor([[generated[-1]]], dtype=torch.long).to(device)
            logits, hidden = model(input_tensor, hidden)
            logits = logits[0, -1, :]
            # Zarządzanie fazami
            tokens_left_in_mode -= 1
            
            if tokens_left_in_mode <= 0:
                
                in_creative_mode = not in_creative_mode
                if in_creative_mode:
                    
                    tokens_left_in_mode = random.randint(100, 200)
                else:
                    
                    tokens_left_in_mode = random.randint(600, 1000)

            # Zarządzanie parametrami fazy
            if in_creative_mode:
                current_temp = temperature + 0.25  
                p = 0.96                         
            else:
                current_temp = temperature 
                p = 0.88 

            # Bezpiecznik perkusyjny
            recent_history = generated[-40:]
            percussion_count = sum(1 for t in recent_history if not is_not_percusion(t))
            
            if percussion_count > 12:  
                for token_idx in range(logits.size(-1)):
                    if not is_not_percusion(token_idx):
                        logits[token_idx] -= 3.0 

            # Kara na powtarzające się nuty
            if not in_creative_mode:
                for token in set(recent_history):
                    if is_note(token):
                        count = recent_history.count(token)
                        if count > 3: 
                            logits[token] -= (0.8 * count)

            # Blokada ostatniego programu
            logits[last_active_program] -= 12.0
            
            # Sampling
            logits = logits / current_temp
            
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

            sorted_indices_to_remove = cumulative_probs > p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            logits[indices_to_remove] = float('-inf')

            probabilities = F.softmax(logits, dim=-1)
            next_token_id = torch.multinomial(probabilities, num_samples=1).item()
            
            if 'PROGRAM' in id2token[next_token_id]:
                last_active_program = next_token_id

            generated.append(next_token_id)
    
            if next_token_id == token2id.get('EOS'):
                break
                
    generated_tokens = [id2token[idx] for idx in generated]
    return generated_tokens


In [437]:
# Generacja muzyki z modyfikacjami
def generate_music(model, token2id, id2token, max_generated_tokens=2000, temperature=0.9):
    model.eval()

    last_detected_pattern = None
    pattern_repeat_count = 0
    N = 6

    generated = [token2id['BOS']]
    
    hidden = None
    change_point = random.randint(2000,4000)

    with torch.no_grad():
        for i in range(max_generated_tokens):

            
            input_tensor = torch.tensor([[generated[-1]]], dtype=torch.long).to(device)
            
            logits, hidden = model(input_tensor, hidden)
            
            logits = logits[0, -1, :]

            # Nakładanie kary na powtarzające się nuty
            recent_window = generated[-50:]
            for token in set(recent_window):
                count = recent_window.count(token)
                if count > 2: 
                    logits[token] -= (0.5 * count)

            # wykrywanie sekwencji
            #melody_only_history = [t for t in generated if is_note(t)]

            #if len(melody_only_history) >= N * 2:
                #current_pattern = melody_only_history[-N:]
                #previous_pattern = melody_only_history[-2*N:-N]
                
                #if current_pattern == previous_pattern:
                    #if current_pattern == last_detected_pattern:
                        #pattern_repeat_count += 1
                    #else:
                        #last_detected_pattern = current_pattern
                        #pattern_repeat_count = 2
                #else:
                    #pattern_repeat_count = 0
                    #last_detected_pattern = None

                #if pattern_repeat_count >= 4:
                    #forbidden_token = melody_only_history[-N]
                    #logits[forbidden_token] -= 4.0
            
            # Wyeliminowanie preferowanego instrumentu na starcie, by zwiększyć różnorodność
            # Bez tego większość piosenek zaczyna się identycznie i jest bardzo podobna przez całą resztę trwania melodii
            #if len(generated) < 20:
                #logits[417] = float('-inf')

            # Zablokowanie pętli perkusyjnej
            recent_history = generated[-30:]
            percussion_count = sum(1 for t in recent_history if not is_not_percusion(t))
            
            if percussion_count > 12:  # Obniżony próg dla szybszej reakcji
                for token_idx in range(logits.size(-1)):
                    if not is_not_percusion(token_idx):
                        logits[token_idx] -= 2.5

            # Dynamiczna zmiana temperatury w celu zwiększenia różnorodności
            if change_point <= 0:
                current_temp = temperature + 0.2  
                if change_point < -10:            
                    change_point = random.randint(2000, 3000)
            else:
                current_temp = temperature

            # część funckji bazowej
            logits = logits / current_temp
            
            # Dynamiczny sampling
            p = 0.96 if change_point <= 0 else 0.90
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

            sorted_indices_to_remove = cumulative_probs > p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            logits[indices_to_remove] = float('-inf')

            # Ciąg dalszy normalnej generacji
            probabilities = F.softmax(logits, dim=-1)
            
            next_token_id = torch.multinomial(probabilities, num_samples=1).item()
            
            generated.append(next_token_id)

            change_point -= 1
    
            if next_token_id == token2id.get('EOS'):
                break
                
    generated_tokens = [id2token[idx] for idx in generated]
    return generated_tokens




In [438]:
# Generacja muzyki
def generate_music_clear(model, token2id, id2token, max_generated_tokens=2000, temperature=0.9):
    model.eval()

    last_detected_pattern = None
    pattern_repeat_count = 0
    N = 3

    generated = [token2id['BOS']]
    
    hidden = None
    change_point = random.randint(560,1000)

    with torch.no_grad():
        for i in range(max_generated_tokens):

            
            input_tensor = torch.tensor([[generated[-1]]], dtype=torch.long).to(device)
            
            logits, hidden = model(input_tensor, hidden)
            
            logits = logits[0, -1, :]

            # część funckji bazowej
            logits = logits / temperature

            # Ciąg dalszy normalnej generacji
            probabilities = F.softmax(logits, dim=-1)
            
            next_token_id = torch.multinomial(probabilities, num_samples=1).item()
            
            generated.append(next_token_id)

            change_point -= 1
    
            if next_token_id == token2id.get('EOS'):
                break
                
    generated_tokens = [id2token[idx] for idx in generated]
    return generated_tokens



In [472]:
# Generowanie utworów
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, FILE_NAME_LOAD)))

MAX_GENERATED_TOKENS = 12400
TEMPERATURE = 0.45

generated_sequence = generate_music_cerative(model, token2id, id2token, max_generated_tokens=MAX_GENERATED_TOKENS, temperature=TEMPERATURE)
print(f"Generated {len(generated_sequence)} tokens.")
print("Example fragment::", generated_sequence[:30])

Generated 12401 tokens.
Example fragment:: ['BOS', 'PROGRAM_34', 'NOTE_ON_26', 'PROGRAM_81', 'NOTE_ON_26', 'PROGRAM_30', 'NOTE_ON_38', 'DRUM_45', 'DRUM_36', 'SHIFT_8', 'NOTE_OFF_26', 'NOTE_OFF_38', 'NOTE_OFF_38', 'PROGRAM_34', 'NOTE_ON_26', 'PROGRAM_81', 'NOTE_ON_38', 'PROGRAM_30', 'NOTE_ON_38', 'DRUM_33', 'DRUM_40', 'DRUM_49', 'SHIFT_8', 'NOTE_OFF_38', 'NOTE_OFF_38', 'PROGRAM_81', 'NOTE_ON_38', 'PROGRAM_30', 'NOTE_ON_38', 'DRUM_36']


In [440]:
# Konwersja tokenów do MIDI
def events_to_midi(events, bpm=95):
    sec_per_step = (60.0 / bpm) / STEPS_PER_BEAT
    midi      = pretty_midi.PrettyMIDI()
    insts     = {}     # program -> Instrument (melodyczne)
    drum_inst = None   # jeden wspólny kanał perkusji
    active    = {}     # pitch -> [(start_step, program)]
    cur, last_prog = 0, 0

    for ev in events:
        if ev in ('BOS', 'EOS', 'PAD'):
            continue
        if ev.startswith('SHIFT_'):
            cur += int(ev.split('_')[1])
        elif ev.startswith('PROGRAM_'):
            last_prog = int(ev.split('_')[1])
        elif ev.startswith('NOTE_ON_'):
            pitch = int(ev.split('_')[-1])
            active.setdefault(pitch, []).append((cur, last_prog))
        elif ev.startswith('NOTE_OFF_'):
            pitch = int(ev.split('_')[-1])
            if active.get(pitch):
                start_step, prog = active[pitch].pop(0)
                start = start_step * sec_per_step
                end   = cur * sec_per_step
                if end > start:
                    if prog not in insts:
                        insts[prog] = pretty_midi.Instrument(program=prog)
                    insts[prog].notes.append(pretty_midi.Note(100, pitch, start, end))
        elif ev.startswith('DRUM_'):
            pitch = int(ev.split('_')[1])
            start = cur * sec_per_step
            end   = start + sec_per_step
            if drum_inst is None:
                drum_inst = pretty_midi.Instrument(program=0, is_drum=True)
            drum_inst.notes.append(pretty_midi.Note(100, pitch, start, end))

    for inst in insts.values():
        midi.instruments.append(inst)
    if drum_inst is not None:
        midi.instruments.append(drum_inst)
    return midi

In [475]:
# Zapis wygenerowanej muzyki do pliku MIDI

VERSION = "V78"

events_to_midi(generated_sequence, bpm = 50).write(os.path.join(OUTPUT_DIR, f"Generated_DOOM_{VERSION}_v1.mid"))
events_to_midi(generated_sequence, bpm = 75).write(os.path.join(OUTPUT_DIR, f"Generated_DOOM_{VERSION}_v2.mid"))
events_to_midi(generated_sequence, bpm = 95).write(os.path.join(OUTPUT_DIR, f"Generated_DOOM_{VERSION}_v3.mid"))
events_to_midi(generated_sequence, bpm = 120).write(os.path.join(OUTPUT_DIR, f"Generated_DOOM_{VERSION}_v4.mid"))
events_to_midi(generated_sequence, bpm = 140).write(os.path.join(OUTPUT_DIR, f"Generated_DOOM_{VERSION}_v5.mid"))
events_to_midi(generated_sequence, bpm = 165).write(os.path.join(OUTPUT_DIR, f"Generated_DOOM_{VERSION}_v6.mid"))
events_to_midi(generated_sequence, bpm = 195).write(os.path.join(OUTPUT_DIR, f"Generated_DOOM_{VERSION}_v7.mid"))